# Pipeline de Processamento de Linguagem Natural (PLN)

O objetivo deste processo não é apenas rodar algoritmos, mas traduzir a linguagem humana (caótica e cheia de ruídos) em uma matriz matemática limpa que um computador consiga agrupar por semelhança.

In [ ]:
import pandas as pd
import spacy
import re
import time
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import KMeans

# Carregamos o modelo em português. 
# OTIMIZAÇÃO: Desativamos o 'parser' (análise sintática de dependência) e o 'ner' (reconhecimento de entidades).
# Como só queremos lematizar e filtrar classes gramaticais, esses componentes só fariam o código perder tempo.
nlp = spacy.load("pt_core_news_sm", disable=["parser", "ner"])

# Simulação de carregamento do DataFrame
df = pd.read_csv('seu_arquivo.csv')

## 1. Limpeza Estrutural (Remoção de Pontuação e Ruído)

**Explicação Intuitiva:** Um computador é burro. Para ele, "Futebol.", "Futebol," e "futebol" são três palavras completamente diferentes. A pontuação e o excesso de formatação criam variáveis falsas na nossa base de dados. Além disso, números e caracteres especiais raramente definem o *tópico* central de um texto. Nosso primeiro passo é passar um trator no texto, deixando apenas letras minúsculas e espaços.

In [ ]:
def limpa_texto_regex(text):
    text = str(text).lower() # Tudo minúsculo
    text = re.sub(r'\W+', ' ', text) # Substitui qualquer coisa que NÃO seja letra/número por espaço
    text = re.sub(r'\d+', '', text) # Remove todos os números
    text = re.sub(r'\b[a-z]\b', '', text) # Remove letras isoladas perdidas (ex: 'o', 'a', 'e')
    return text

# Aplicamos a limpeza rápida via Regex em toda a coluna.
df['texto_limpo'] = df['texto'].apply(limpa_texto_regex)

## 2. Tokenização, Remoção de Stop Words e Lematização

**Explicação Intuitiva:**
- **Tokenização:** É picotar a frase em palavras individuais. A frase vira uma lista de peças de lego.
- **Stop Words:** Palavras como "que", "de", "para", "com" são a argamassa da linguagem. Elas unem as frases, mas não carregam significado temático. Se não as removermos, o K-Means vai agrupar os textos porque todos usam a palavra "que", e não porque falam de esportes ou política.
- **Lematização:** É reduzir a palavra à sua raiz de dicionário. O computador não sabe que "correu", "correndo" e "correrá" são a mesma ação. A lematização transforma todas em "correr". Isso reduz drasticamente o tamanho do nosso vocabulário e concentra a força matemática no real significado.

**Por que usar `nlp.pipe`?** Aplicar isso linha por linha é ineficiente. O `pipe` envia os textos em lotes para a memória, otimizando o processamento sob o capô.

In [ ]:
lemmas_finais = []

# Processamos os textos em lotes de 100.
for doc in nlp.pipe(df['texto_limpo'].tolist(), batch_size=100):
    lemmas = []
    for token in doc:
        # Se NÃO for stop word E for um Substantivo, Verbo ou Adjetivo...
        if not token.is_stop and token.pos_ in ['NOUN', 'VERB', 'ADJ']:
            # ...pegamos o lema da palavra.
            lemmas.append(token.lemma_)
            
    # Juntamos as peças de lego de volta em uma string (frase processada)
    lemmas_finais.append(" ".join(lemmas))

df['texto_processado'] = lemmas_finais

## 3. Modelo Bag of Words (BoW)

**Explicação Intuitiva:** O texto agora está limpo, mas computadores não calculam palavras, calculam números. O BoW cria um enorme "saco de palavras" com todas as palavras únicas que sobraram em toda a base. Em seguida, ele olha para cada texto e conta: quantas vezes a palavra X aparece aqui? A palavra Y? 

O resultado é uma matriz gigantesca onde cada linha é um documento e cada coluna é uma palavra. Para evitar que essa matriz fique grande e inútil (maldição da dimensionalidade), nós limitamos o vocabulário usando parâmetros rigorosos.

In [ ]:
# max_features=3000: Pega apenas as 3000 palavras mais importantes.
# min_df=10: A palavra tem que aparecer em pelo menos 10 documentos (ignora raridades/erros de digitação).
# max_df=0.7: Se a palavra aparece em mais de 70% dos documentos, ela é genérica demais e descartada.
vectorizer = CountVectorizer(max_features=3000, min_df=10, max_df=0.7)

# Transforma os textos lematizados em uma matriz matemática de ocorrências
X_bow = vectorizer.fit_transform(df['texto_processado'])

print(f"Tamanho final da matriz matemática: {X_bow.shape}")

## 4. Clusterização (K-Means)

**Explicação Intuitiva:** Com a matriz pronta, os textos agora são coordenadas em um espaço multidimensional. Textos que usam as mesmas palavras terão coordenadas próximas. O K-Means joga *K* pontos aleatórios (centróides) nesse espaço e começa a puxar os textos mais próximos para formar grupos. 

Ao final, cada centróide ficará no "centro de gravidade" de um assunto. Ao olhar para as palavras que têm mais peso (maior frequência) nas coordenadas desse centróide, nós descobrimos magicamente qual é o tópico daquele grupo, sem ter lido um único texto.

In [ ]:
# Aqui definimos K=5 como exemplo. O ideal é usar o Método do Cotovelo ou Silhouette Score para achar o número exato.
NUM_CLUSTERS = 5
kmeans = KMeans(n_clusters=NUM_CLUSTERS, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_bow)

# Extração dos Tópicos Baseado no Centro de Gravidade (Centróide)
vocabulario = vectorizer.get_feature_names_out()
centroides = kmeans.cluster_centers_

print("Tópicos identificados por grupo:")
for i in range(NUM_CLUSTERS):
    # argsort() ordena os valores do menor para o maior. 
    # O [::-1] inverte para os maiores primeiro. O [:10] pega os 10 primeiros.
    top_palavras_idx = centroides[i].argsort()[::-1][:10]
    palavras = vocabulario[top_palavras_idx]
    print(f"Grupo {i}: {', '.join(palavras)}")